# 24 — Deep-dive ngành ngân hàng

Ngân hàng chiếm phần lớn vốn hoá VNINDEX, và không một chỉ tiêu nào của
screener chung ở notebook `23` chạm được vào thứ quyết định giá trị của chúng.
ROE của một ngân hàng không nói gì về việc nó kiếm tiền bằng cách cho vay rủi
ro hay bằng cách vận hành rẻ.

`indicator_catalog(com_type="NH")` có **43 chỉ tiêu riêng** cho việc này.
Notebook dựng bốn góc nhìn:

1. **Sinh lời**: NIM và phân rã của nó thành YOEA − COF
2. **Chất lượng tài sản**: nợ xấu, nợ nhóm 2 (chỉ báo sớm), bao phủ nợ xấu
3. **Vận hành và nguồn vốn**: CIR, LDR, tỷ trọng vốn bán buôn
4. **Bảng điểm tổng hợp** cho toàn ngành

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, heatmap, hom_nay, lui_ngay
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Vũ trụ ngân hàng

Ngành ngân hàng là ICB `8300` ở cấp 2. Dùng `meta.symbols(icb=...)` — nó nhận
cả mã cấp 2 lẫn cấp 4 nên bạn không cần biết mình đang cầm cấp nào.

In [2]:
ngan_hang = client.meta.symbols(icb="8300")

print(f"{len(ngan_hang)} ngân hàng niêm yết:")
print(ngan_hang.groupby("exchange", observed=True)["symbol"].agg(["count", lambda s: ", ".join(sorted(s))]).to_string())

MA_NH = ngan_hang["symbol"].tolist()

28 ngân hàng niêm yết:
          count                                                                                                    <lambda_0>
exchange                                                                                                                     
HNX           2                                                                                                      BAB, NVB
HOSE         22  ACB, BID, BVB, CTG, EIB, HDB, KLB, LPB, MBB, MSB, NAB, OCB, SHB, SSB, STB, TCB, TPB, VAB, VBB, VCB, VIB, VPB
UPCOM         4                                                                                            ABB, PCB, PGB, SGB


Cả 28 mã đều là `company_type == "NH"` — điều này quan trọng vì nó nghĩa là
toàn bộ 43 chỉ tiêu riêng của ngân hàng đều dùng được cho cả nhóm:

In [3]:
print(ngan_hang["company_type"].value_counts().to_string())

company_type
NH    28


## 2 · Bộ chỉ tiêu riêng của ngân hàng

Đọc `formula` trước khi dùng. Bốn dòng dưới đây là những chỗ một giả định sai
dẫn tới kết luận ngược:

In [4]:
cat_nh = client.financials.indicator_catalog(com_type="NH")
pd.set_option("display.max_colwidth", 130)

for ma_ct in ["nim", "npl_ratio", "group2_ratio", "npl_coverage", "ldr", "car"]:
    r = cat_nh[cat_nh["code"] == ma_ct].iloc[0]
    print(f"■ {r['label']} — {ma_ct} [{r['unit']}] · cao hơn tốt hơn: {r['higher_is_better']}")
    print(f"  {r['formula']}\n")

■ NIM — biên lãi thuần — nim [ratio] · cao hơn tốt hơn: True
  Thu nhập lãi thuần ÷ tài sản sinh lãi bình quân.

■ Tỷ lệ nợ xấu — npl_ratio [ratio] · cao hơn tốt hơn: False
  Nợ nhóm 3, 4, 5 ÷ tổng dư nợ cho vay khách hàng. Ngưỡng quy định khoảng 3%.

■ Tỷ lệ nợ nhóm 2 — group2_ratio [ratio] · cao hơn tốt hơn: False
  Nợ cần chú ý (nhóm 2) ÷ tổng dư nợ cho vay khách hàng. Đây là chỉ báo **sớm**: nợ nhóm 2 hôm nay là nợ xấu tiềm tàng của vài quý sau.

■ Bao phủ nợ xấu — npl_coverage [x] · cao hơn tốt hơn: True
  Dự phòng rủi ro cho vay ÷ nợ xấu. Trên 1 nghĩa là đã trích đủ để xoá toàn bộ nợ xấu hiện có.

■ LDR — cho vay trên tiền gửi — ldr [ratio] · cao hơn tốt hơn: False
  Dư nợ cho vay khách hàng ÷ tiền gửi khách hàng.

■ CAR — hệ số an toàn vốn — car [ratio] · cao hơn tốt hơn: True
  Vốn tự có ÷ tài sản có rủi ro quy đổi, theo công bố của ngân hàng. ⚠️ Nhiều ngân hàng không công bố đều đặn; rỗng nghĩa là **không công bố**, không phải bằng 0.



Hai điều đáng chú ý ngay:

- **`group2_ratio` là chỉ báo sớm.** Nợ nhóm 2 hôm nay là nợ xấu của vài quý
  tới. Nhìn `npl_ratio` một mình là nhìn vào quá khứ.
- **`car` không phải ai cũng công bố.** Nó là số ngân hàng tự báo, không phải
  số tính ra từ báo cáo — nên độ phủ thấp hơn hẳn các chỉ tiêu khác.

## 3 · Lấy dữ liệu

⚠️ Dùng `period="annual"`. Notebook `22` đã đo: với loại hình `NH`, chỉ tiêu
quarterly đã được quy về năm, còn với `CT` thì không — nên nếu sau này bạn ghép
bảng này với dữ liệu doanh nghiệp phi tài chính, kỳ `annual` là mặt bằng chung
duy nhất an toàn.

In [5]:
NHOM_CHI_TIEU = ["profitability", "asset_quality", "efficiency", "liquidity", "capital", "income_structure", "size"]

thu = client.financials.indicators(MA_NH[:10], codes=["roe"], period="annual", start_year=HOM_NAY.year - 2)
do_phu = thu.groupby("year", observed=True)["symbol"].nunique()
NAM = int(do_phu[do_phu >= do_phu.max() * 0.9].index.max())
print(f"Năm gần nhất có số liệu đầy đủ: {NAM}")

raw = client.financials.indicators(
    MA_NH, groups=NHOM_CHI_TIEU, period="annual", start_year=NAM, end_year=NAM
)
nh = raw.pivot_table(index="symbol", columns="code", values="value").join(
    ngan_hang.set_index("symbol")[["short_name", "exchange"]]
)

print(f"\n{len(nh)} ngân hàng × {raw['code'].nunique()} chỉ tiêu")
print("\nĐộ phủ các chỉ tiêu dưới 100%:")
phu = (nh[raw["code"].unique()].notna().sum() / len(nh) * 100).round(0)
print(phu[phu < 100].astype(int).astype(str).add("%").to_string())

Năm gần nhất có số liệu đầy đủ: 2025



28 ngân hàng × 31 chỉ tiêu

Độ phủ các chỉ tiêu dưới 100%:
car    79%


Chỉ `car` là thiếu — đúng như cảnh báo trong `formula`. Mọi chỉ tiêu tính ra
được từ báo cáo tài chính đều đủ 100%.

## 4 · Sinh lời: NIM và phân rã YOEA − COF

NIM là biên lãi thuần — chênh lệch giữa lãi thu vào trên tài sản sinh lãi và
lãi trả ra trên nguồn vốn. Nhìn NIM một mình không biết ngân hàng đang **cho
vay giá cao** hay **huy động giá rẻ**; hai chiến lược khác nhau hoàn toàn về
rủi ro.

```
spread  =  YOEA        −  COF
           (lợi suất       (chi phí
            tài sản)        vốn)
```

In [6]:
sinh_loi = nh[["short_name", "nim", "yoea", "cof", "spread", "roe", "roa"]].dropna(subset=["nim"])
sinh_loi_pct = sinh_loi.copy()
for c in ["nim", "yoea", "cof", "spread", "roe", "roa"]:
    sinh_loi_pct[c] = (sinh_loi_pct[c] * 100).round(2)

sinh_loi_pct.sort_values("nim", ascending=False).head(12)

,short_name,nim,yoea,cof,spread,roe,roa
symbol,,,,,,,
VPB,VPBank,5.67,9.79,4.77,5.01,14.65,2.20
HDB,HDBank,4.72,9.23,4.60,4.63,24.49,2.03
KLB,KienlongBank,4.22,9.41,5.32,4.08,24.71,1.90
MBB,MBBank,3.93,6.78,3.17,3.61,20.67,1.95
TCB,Techcombank,3.80,6.76,3.34,3.43,15.45,2.33
STB,NH Sài Gòn Tài Lộc (SACOMBANK),3.31,7.14,4.08,3.06,10.34,0.71
OCB,Ngân hàng Phương Đông,3.24,7.39,4.63,2.75,12.23,1.33
MSB,MSB Bank,3.18,6.27,3.35,2.92,14.20,1.55
VIB,VIBBank,3.14,7.08,4.31,2.77,16.42,1.39


In [7]:
top = sinh_loi.nlargest(15, "nim")

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=top.index,
        y=top["yoea"] * 100,
        name="YOEA — lợi suất tài sản sinh lãi",
        marker=dict(color=CHUOI[0], line=dict(color="#fcfcfb", width=2)),
        hovertemplate="%{x}: %{y:.2f}%<extra>YOEA</extra>",
    )
)
fig.add_trace(
    go.Bar(
        x=top.index,
        y=-top["cof"] * 100,
        name="COF — chi phí vốn (vẽ xuống dưới)",
        marker=dict(color=CHUOI[1], line=dict(color="#fcfcfb", width=2)),
        hovertemplate="%{x}: %{y:.2f}%<extra>COF</extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=top.index,
        y=top["nim"] * 100,
        name="NIM",
        mode="markers+text",
        marker=dict(size=11, color=CHUOI[5], line=dict(color="#fcfcfb", width=2)),
        text=[f"{v * 100:.2f}" for v in top["nim"]],
        textposition="top center",
        textfont=dict(size=10, color="#52514e"),
        hovertemplate="%{x}: %{y:.2f}%<extra>NIM</extra>",
    )
)
fig.add_hline(y=0, line_width=1, line_color="#c3c2b7")
fig.update_layout(
    barmode="relative",
    title_text=f"Cấu trúc biên lãi — 15 ngân hàng NIM cao nhất, năm {NAM}<br>"
    "<sub style='color:#52514e'>Cột lên là lãi thu vào, cột xuống là lãi trả ra, chấm là biên còn lại</sub>",
    yaxis_title="% trên tài sản sinh lãi / nguồn vốn",
    height=520,
)
fig

Biểu đồ này tách hai chiến lược: ngân hàng có cột xanh cao (cho vay lợi suất
cao — thường là bán lẻ, tiêu dùng, rủi ro cao hơn) so với ngân hàng có cột cam
ngắn (huy động rẻ — thường là CASA lớn, tệp khách hàng doanh nghiệp lớn).

⚠️ NIM cao **không tự nó là tốt**. Nó phải được đọc cùng chất lượng tài sản ở
phần sau — lợi suất cho vay cao thường đi kèm nợ xấu cao, và đó là một sự đánh
đổi chứ không phải một bữa trưa miễn phí.

## 5 · Chất lượng tài sản

Ba con số, đọc cùng nhau mới có nghĩa:

| Chỉ tiêu | Câu hỏi |
|---|---|
| `npl_ratio` | Nợ xấu hiện tại bao nhiêu? (ngưỡng quy định ~3%) |
| `group2_ratio` | Bao nhiêu nợ **sắp** thành nợ xấu? |
| `npl_coverage` | Đã trích dự phòng đủ để xoá hết nợ xấu chưa? (>1 là đủ) |

In [8]:
chat_luong = nh[["short_name", "npl_ratio", "group2_ratio", "group5_share", "npl_coverage", "llr_to_loan"]].dropna(
    subset=["npl_ratio"]
)
cl_pct = chat_luong.copy()
for c in ["npl_ratio", "group2_ratio", "group5_share", "llr_to_loan"]:
    cl_pct[c] = (cl_pct[c] * 100).round(2)
cl_pct["npl_coverage"] = cl_pct["npl_coverage"].round(2)

cl_pct.sort_values("npl_ratio").rename(
    columns={
        "npl_ratio": "NPL %",
        "group2_ratio": "Nhóm 2 %",
        "group5_share": "Nhóm 5 / NPL %",
        "npl_coverage": "Bao phủ (lần)",
        "llr_to_loan": "Dự phòng / dư nợ %",
    }
)

,short_name,NPL %,Nhóm 2 %,Nhóm 5 / NPL %,Bao phủ (lần),Dự phòng / dư nợ %
symbol,,,,,,
VCB,Vietcombank,0.58,0.16,89.83,2.58,1.49
ABB,Ngân hàng An Bình,0.88,0.76,57.07,1.20,1.05
ACB,ACB,0.97,0.36,76.99,1.14,1.11
TCB,Techcombank,1.07,0.50,72.47,1.28,1.37
CTG,VietinBank,1.10,0.87,90.56,1.59,1.75
BAB,Ngân hàng Bắc Á,1.15,0.29,81.11,1.08,1.24
MBB,MBBank,1.29,0.94,45.78,0.94,1.21
TPB,TPBank,1.29,1.65,34.89,0.92,1.19
VAB,Ngân hàng Việt Á,1.31,0.02,56.03,0.91,1.20


### Nợ xấu đối chiếu mức bao phủ

Góc phần tư quan trọng nhất trong phân tích ngân hàng: nợ xấu thấp **và** dự
phòng dày là an toàn; nợ xấu cao **và** dự phòng mỏng là chỗ rủi ro nằm.

In [9]:
ve = chat_luong.dropna(subset=["npl_coverage"]).copy()
nguong_npl = 3.0  # ngưỡng quy định, %

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ve["npl_ratio"] * 100,
        y=ve["npl_coverage"],
        mode="markers+text",
        marker=dict(
            size=11,
            color=CHUOI[0],
            opacity=0.75,
            line=dict(width=2, color="#fcfcfb"),
        ),
        text=ve.index,
        textposition="top center",
        textfont=dict(size=10, color="#52514e"),
        hovertemplate="<b>%{text}</b><br>NPL %{x:.2f}%<br>Bao phủ %{y:.2f} lần<extra></extra>",
        showlegend=False,
    )
)
fig.add_hline(y=1, line_dash="dot", line_width=1, line_color="#898781",
              annotation_text="bao phủ 1,0 lần", annotation_position="right")
fig.add_vline(x=nguong_npl, line_dash="dot", line_width=1, line_color="#898781",
              annotation_text=f"NPL {nguong_npl:.0f}%", annotation_position="top")
fig.add_annotation(x=ve["npl_ratio"].min() * 100, y=ve["npl_coverage"].max(),
                   text="nợ xấu thấp · dự phòng dày", showarrow=False,
                   font=dict(color=TANG, size=12), xanchor="left")
fig.update_layout(
    title_text=f"Nợ xấu và mức bao phủ — năm {NAM}<br>"
    "<sub style='color:#52514e'>Góc trên bên trái là an toàn nhất; góc dưới bên phải là chỗ rủi ro tập trung</sub>",
    xaxis_title="Tỷ lệ nợ xấu (%)",
    yaxis_title="Bao phủ nợ xấu (lần)",
    height=560,
)
fig

### Nợ nhóm 2 — nhìn về phía trước

Nợ nhóm 2 là nợ quá hạn 10–90 ngày. Nó chưa vào NPL nhưng phần lớn sẽ vào.
Ngân hàng có nhóm 2 cao hơn NPL nhiều lần là ngân hàng có NPL đang trên đường
tăng.

In [10]:
canh_bao = chat_luong.assign(ty_le_nhom2_tren_npl=lambda d: d["group2_ratio"] / d["npl_ratio"]).dropna()

bar_ngang(
    canh_bao.reset_index().assign(nhan=lambda d: d["symbol"]),
    nhan="nhan",
    gia_tri="ty_le_nhom2_tren_npl",
    tieu_de="Nợ nhóm 2 chia nợ xấu",
    phu_de="Trên 1 nghĩa là lượng nợ 'sắp xấu' đang lớn hơn lượng nợ đã xấu",
    nhan_x="lần",
    dinh_dang_nhan="{:.2f}×",
)

⚠️ Biểu đồ này dùng hàm tô màu theo dấu, mà mọi giá trị đều dương nên tất cả
đều xanh — màu ở đây **không mang thông tin**. Con số trên thanh và độ dài thanh
mới mang. Đây chính là lý do mọi hàm vẽ trong bộ này luôn in số ra: khi màu
không nói gì, số vẫn nói.

## 6 · Vận hành và nguồn vốn

In [11]:
van_hanh = nh[["short_name", "cir", "cost_to_asset", "ldr", "wholesale_funding_ratio",
               "non_interest_income_ratio", "equity_to_assets"]].dropna(subset=["cir"])

vh = van_hanh.copy()
for c in vh.columns.drop("short_name"):
    vh[c] = (vh[c] * 100).round(2)

vh.sort_values("cir").rename(
    columns={
        "cir": "CIR %",
        "cost_to_asset": "Chi phí / TS %",
        "ldr": "LDR %",
        "wholesale_funding_ratio": "Vốn bán buôn %",
        "non_interest_income_ratio": "Thu ngoài lãi %",
        "equity_to_assets": "VCSH / TS %",
    }
).head(12)

,short_name,CIR %,Chi phí / TS %,LDR %,Vốn bán buôn %,Thu ngoài lãi %,VCSH / TS %
symbol,,,,,,,
BAB,Ngân hàng Bắc Á,-57.49,-1.28,99.51,28.76,14.60,6.80
SHB,SHB,22.13,0.78,107.46,27.62,30.34,7.64
VPB,VPBank,24.96,1.71,150.29,39.05,21.42,14.31
VAB,Ngân hàng Việt Á,25.68,0.75,89.57,22.69,10.17,7.23
HDB,HDBank,27.17,1.42,97.44,32.34,18.62,8.41
LPB,LPBank,28.29,1.11,116.04,38.31,26.72,7.79
MBB,MBBank,29.07,1.43,117.65,34.38,23.76,8.79
CTG,VietinBank,30.42,1.03,111.07,29.10,23.87,6.49
TCB,Techcombank,30.78,1.51,124.03,37.07,28.54,15.05


**CIR** (chi phí trên thu nhập) thấp là vận hành hiệu quả. **LDR** cao là dùng
hết nguồn tiền gửi để cho vay — sinh lời cao hơn nhưng đệm thanh khoản mỏng
hơn. **Vốn bán buôn** cao nghĩa là phụ thuộc nguồn liên ngân hàng và trái
phiếu, vốn đắt hơn và bốc hơi nhanh hơn tiền gửi dân cư khi thị trường căng.

In [12]:
bang_vh = van_hanh[["cir", "ldr", "wholesale_funding_ratio", "non_interest_income_ratio"]].mul(100).round(1)
bang_vh.columns = ["CIR", "LDR", "Vốn bán buôn", "Thu ngoài lãi"]
bang_vh = bang_vh.loc[bang_vh["CIR"].sort_values().index]

heatmap(
    bang_vh,
    tieu_de=f"Hiệu quả vận hành và cấu trúc nguồn vốn — năm {NAM}",
    phu_de="Xếp theo CIR tăng dần · thang một sắc vì đây là độ lớn, không phải dấu",
    nhan_mau="%",
    phan_ky=False,
    dinh_dang_o="%{z:.0f}",
)

## 7 · Bảng điểm toàn ngành

Ghép năm trục lại. Lần này **không cần chuẩn hoá trong ngành** — cả 28 mã đã ở
cùng một ngành, đó chính là điều kiện làm cho việc so sánh trực tiếp có nghĩa.

In [13]:
TRUC_NH = {
    "Sinh lời": ["roe", "nim"],
    "Chất lượng TS": ["npl_ratio", "group2_ratio", "npl_coverage"],
    "Hiệu quả": ["cir", "cost_to_asset"],
    "Thanh khoản": ["ldr", "wholesale_funding_ratio"],
    "Vốn": ["equity_to_assets"],
}
MA_DUNG = [m for ds in TRUC_NH.values() for m in ds]

chieu = cat_nh.set_index("code")["higher_is_better"]
thieu_chieu = [m for m in MA_DUNG if pd.isna(chieu.get(m))]
assert not thieu_chieu, f"Không xác định được chiều: {thieu_chieu}"


def z_co_dau(s: pd.Series, ma_chi_tieu: str) -> pd.Series:
    """Z-score sau khi cắt đuôi, đảo dấu theo `higher_is_better` của danh mục."""
    kep = s.clip(s.quantile(0.05), s.quantile(0.95))
    sd = kep.std()
    z = pd.Series(0.0, index=s.index) if not np.isfinite(sd) or sd == 0 else (kep - kep.mean()) / sd
    return z if chieu[ma_chi_tieu] else -z


diem = pd.DataFrame(index=nh.index)
for ten_truc, ma_ds in TRUC_NH.items():
    cot = [z_co_dau(nh[m], m) for m in ma_ds if nh[m].notna().sum() > 3]
    diem[ten_truc] = pd.concat(cot, axis=1).mean(axis=1, skipna=True)

diem["Điểm"] = diem.mean(axis=1, skipna=True)
diem = diem.join(nh[["short_name"]]).sort_values("Điểm", ascending=False)

diem.round(2)

,Sinh lời,Chất lượng TS,Hiệu quả,Thanh khoản,Vốn,Điểm,short_name
symbol,,,,,,,
TCB,0.65,1.13,0.05,-0.83,2.31,0.66,Techcombank
VCB,-0.00,1.47,0.46,0.70,0.37,0.60,Vietcombank
VAB,-0.15,0.79,1.36,1.09,-0.52,0.51,Ngân hàng Việt Á
MBB,1.17,0.58,0.24,-0.48,0.19,0.34,MBBank
KLB,1.69,0.29,-1.02,0.84,-0.11,0.34,KienlongBank
ACB,0.25,1.07,0.51,-0.52,0.38,0.34,ACB
SHB,0.21,0.15,1.43,0.23,-0.34,0.33,SHB
HDB,1.90,-0.96,0.33,0.31,0.01,0.32,HDBank
CTG,0.35,1.20,0.77,0.03,-0.86,0.30,VietinBank


In [14]:
bang_diem = diem.head(15)[list(TRUC_NH)].round(2)
bang_diem.index = diem.head(15)["short_name"].str.slice(0, 24)

heatmap(
    bang_diem,
    tieu_de=f"Bảng điểm ngân hàng — năm {NAM}",
    phu_de="Z-score trong chính ngành ngân hàng · dương là tốt hơn trung bình ngành",
    nhan_mau="z-score",
    dinh_dang_o="%{z:+.2f}",
)

## 8 · Diễn biến theo thời gian

Một lát cắt một năm không cho biết ngân hàng đang cải thiện hay xấu đi. Bốn
ngân hàng lớn nhất theo tổng tài sản, năm năm:

In [15]:
LON_NHAT = nh.nlargest(4, "total_assets").index.tolist()
print(f"Bốn ngân hàng lớn nhất theo tổng tài sản: {LON_NHAT}")

lich_su = client.financials.indicators(
    LON_NHAT, codes=["nim", "npl_ratio", "cir", "roe"], period="annual", start_year=NAM - 5
)

for ma_ct, nhan, don_vi in [
    ("nim", "NIM", "%"),
    ("npl_ratio", "Tỷ lệ nợ xấu", "%"),
    ("cir", "CIR — chi phí trên thu nhập", "%"),
]:
    d = lich_su[lich_su["code"] == ma_ct].assign(pct=lambda x: x["value"] * 100)
    fig = duong(
        d,
        x="year",
        y="pct",
        theo="symbol",
        tieu_de=f"{nhan} — bốn ngân hàng lớn nhất",
        nhan_y=don_vi,
    )
    fig.show()

Bốn ngân hàng lớn nhất theo tổng tài sản: ['BID', 'CTG', 'VCB', 'MBB']


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Toàn bộ ngân hàng niêm yết | `meta.symbols(icb="8300")` |
| 43 chỉ tiêu riêng của ngân hàng | `financials.indicator_catalog(com_type="NH")` |
| Cả nhóm chỉ tiêu cùng lúc | `groups=["asset_quality", "profitability", ...]` |

**Bốn điều đáng nhớ:**

1. **NIM cao không tự nó là tốt.** Đọc nó cùng `npl_ratio` — lợi suất cho vay
   cao thường đi kèm rủi ro tín dụng cao.
2. **`group2_ratio` nhìn về phía trước, `npl_ratio` nhìn về phía sau.** Nợ nhóm
   2 hôm nay phần lớn là nợ xấu vài quý tới.
3. **`car` là số ngân hàng tự công bố**, không phải số tính từ báo cáo — độ phủ
   thấp hơn hẳn, và `formula` trong danh mục nói rõ điều đó.
4. Trong một ngành đồng nhất thì z-score trực tiếp là hợp lệ. Chuẩn hoá theo
   ngành ở notebook `23` tồn tại chính vì ở đó **không** có tính đồng nhất ấy.

---

**Track 2 hết.** Tiếp theo là Track 3:
[`31_chi_bao_ky_thuat.ipynb`](../03-phan-tich-ky-thuat/31_chi_bao_ky_thuat.ipynb) — hai tầng chỉ báo, và
bốn cái bẫy chung cho cả hai.